<a href="https://colab.research.google.com/github/noailabs/llm-cli/blob/main/llm_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"]=userdata.get('OPENROUTER_API_KEY')

In [12]:
%%writefile llm-cli.py
#from huggingface_hub import InferenceClient
from openai import OpenAI
import json
import sys
import argparse
import os

def load_env_file(filepath='.env'):
    """
    Load variables from a .env file and return them as a dictionary.

    Args:
        filepath (str): Path to the .env file. Defaults to '.env' in current directory.

    Returns:
        dict: Dictionary containing the key-value pairs from the .env file
    """
    env_vars = {}

    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                # Skip empty lines and comments
                if not line or line.startswith('#'):
                    continue

                # Split on first occurrence of '='
                if '=' in line:
                    key, value = line.split('=', 1)
                    key = key.strip()
                    value = value.strip()

                    # Remove surrounding quotes if present
                    if len(value) > 1 and (
                        (value.startswith('"') and value.endswith('"')) or
                        (value.startswith("'") and value.endswith("'"))
                    ):
                        value = value[1:-1]

                    env_vars[key] = value
    except FileNotFoundError:
        print(f"Warning: {filepath} not found.")
    except Exception as e:
        print(f"Error loading .env file: {e}")

    return env_vars

#env = load_env_file()
#token=env["HF_TOKEN"]
#OPENROUTER_API_KEY=env["OPENROUTER_API_KEY"]
OPENROUTER_API_KEY=os.environ["OPENROUTER_API_KEY"]

# print(token)

#model="deepseek-ai/DeepSeek-V3-0324"
model="deepseek/deepseek-chat-v3-0324:free"

provider="sambanova"

settings={
    "temperature":1.0,
    "top_p":1.0,
}

stream=True





parser = argparse.ArgumentParser(description="Process a prompt for an AI model.")

parser.add_argument(
        "-v", "--verbose",
        required=False,
        help="The verbose output."
)


parser.add_argument(
        "-p", "--prompt",
        type=str,
        required=False,
        help="The prompt to send to the AI model."
)

parser.add_argument(
        "-pf", "--prompt_file",
        type=str,
        required=False,
        help="The promptfile to send to the AI model."
)


parser.add_argument(
        "-mf", "--messages_file",
        type=str,
        required=False,
        help="The messages file."
)


parser.add_argument(
        "-of", "--output_file",
        type=str,
        required=False,
        help="The output file."
)


args = parser.parse_args()

verbose=args.verbose
prompt=args.prompt
prompt_file=args.prompt_file
if prompt_file!=None:
  with open(prompt_file, "r") as file:
     prompt = file.read()

if prompt==None:
    print(f"Error: no prompt or prompt_file.")
    sys.exit(0)

print("Prompt:",prompt)


messages_file=args.messages_file
if messages_file==None:
  messages_file="messages.json"
output_file=args.output_file

# prompt="tell a joke about a corner"

#SYSTEM_PROMPT="""You are code generation assistant.
#Don't add any comments before or after code, you can put it inside as comments blocks.
#Don't start or end with ```, just generate code."""

SYSTEM_PROMPT="You are useful assistant."


messages=[{"role":"system","content":SYSTEM_PROMPT}]

try:
    with open(messages_file, "r") as json_file:
        messages = json.load(json_file)

    print("\nLoaded messages:",len(messages))
    if verbose:
      for msg in messages:
        print(f"{msg['role']}: {msg['content']}")
      print("+-----+")
    print()
except FileNotFoundError:
    print(f"Warning: {messages_file} not found.")
except json.JSONDecodeError:
    print(f"Error: Invalid JSON in {messages_file}.")
    sys.exit(0)

messages.append({"role": "user","content": prompt})


client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENROUTER_API_KEY,
)


#client = InferenceClient(
#    token=token,
#    provider=provider, #"hyperbolic", #"together",
#    model=model,
#    #api_key="my_together_api_key",
#)
#resp=client.chat_completion(messages, max_tokens=1000)



response=""
if stream:
  completion = client.chat.completions.create(
    model=model,
    messages=messages,
    stream=True,
    **settings,
  )
  for chunk in completion:
    content = chunk.choices[0].delta.content
    if content is not None:
        response+=content
        sys.stdout.write(content)
        sys.stdout.flush()
  print()
else:
  completion = client.chat.completions.create(
    model=model,
    messages=messages,
    stream=False,
    **settings,
  )
  response=completion.choices[0].message.content
  print(response)

messages.append({"role": "assistant", "content": response})



if output_file!=None:
  with open(output_file, 'w') as file:
    file.write(response)


with open(messages_file, "w") as json_file:
    json.dump(messages, json_file, indent=4)

Overwriting llm-cli.py


In [8]:
!python llm-cli.py -p "tell a joke about a  cat!"


Loaded messages: 5

Here’s a purr-fectly silly cat joke for you:  

---  

**Why did the cat sit on the computer?**  

*Because it wanted to keep an eye on the mouse!*  

---  

(Extra groan-worthy bonus: **What do you call a cat that loves bowling?** *An alley cat!*) 😸🎳


In [10]:
%%writefile p1.txt
Tell a joke about someting wonderful.


Writing p1.txt


In [15]:
!python llm-cli.py -pf p1.txt -m m1.json

Prompt: Tell a joke about someting wonderful.


Loaded messages: 3

Of course! Here’s a wholesome joke about something universally wonderful:  

**Why did the baby cookie bring a ladder to the milk glass?**  

*Because it wanted to be a **dunk** champion!*  

(And honestly, isn’t the idea of cookies and milk teaming up just *wonderful*?)  

Want something even sweeter? Let me know! 😄🍪🥛
